# Lab 6 - Sampling & Partitioning

In this lab, we will explore different methods for data sampling and partitioning.

In [1]:
import sys
!"{sys.executable}" -m pip install pandas
!"{sys.executable}" -m pip install numpy
!"{sys.executable}" -m pip install scikit-learn
import pandas as pd
import numpy as np
import sklearn

We will first import the Water Potability dataset. It can be found on the Moodle.

In [ ]:
water_df = pd.read_csv('Watera.csv')
print(water_df.columns)

Index(['ph', 'hardness', 'tds', 'chlorine', 'sulfate', 'conductivity',
       'organic_carbon', 'trihalomethanes', 'turbidity', 'potability'],
      dtype='str')


We can use value_counts to see how many examples we have of each class. sort_index shows the classes in numerical order. normalize=True*100 converts the relative counts to a percentage.

In [ ]:
# 选择potability列，统计每个类别的比例
# normalize=True：返回比例，而不是数量
# sort_index()：按照类别值0、1升序排列
# *100：将比例转换为百分比
# print()：输出最终结果
print(water_df["potability"].value_counts(normalize=True).sort_index()*100)

potability
0    92.384
1     7.616
Name: proportion, dtype: float64


We can see that our dataset is highly imbalanced, with roughly 92% of examples coming from one of the two classes. Let's sample our dataset at random, and then look at the value counts:

In [ ]:
# 从water_df中随机抽取250条记录
# random_state=2：固定随机种子，使每次运行得到相同的样本
# 默认不放回抽样，即同一条记录不会被重复抽取
water_df_sampled = water_df.sample(n=250, random_state=2)

# 统计抽样后potability各类别的比例
# normalize=True：计算各类别占比
# sort_index()：按照类别标签0、1升序排列
# *100：将比例转换为百分数
print(water_df_sampled["potability"].value_counts(normalize=True).sort_index()*100)

potability
0    89.6
1    10.4
Name: proportion, dtype: float64


We can see that our data is still highly imbalanced (as we would expect), with around 90% of objects coming from a single class.

Let's run this experiment 50 times and see what the distribution looks like per sample:

In [ ]:
# 创建空列表，用于保存每次抽样后类别0所占的百分比
sampled_y = []

# 使用1到50作为随机种子，进行50次独立设置种子的随机抽样
for seed in range(1,51):

    # 每次随机抽取250条记录
    # 计算potability各类别的百分比，并按类别标签排序
    # iloc[0]取排序后的第一个类别的百分比，通常为类别0
    # append()将本次抽样结果添加到列表
    sampled_y.append((water_df.sample(n=250, random_state=seed)["potability"].value_counts(normalize=True).sort_index()*100).iloc[0])

# min()找出50次抽样中类别0的最低百分比
# max()找出50次抽样中类别0的最高百分比
# round(..., 2)将数值四舍五入，保留两位小数
print("Minimum class 0 proportion:", round(min(sampled_y),2), ", Maximum class 0 proportion:", round(max(sampled_y),2))

Minimum class 0 proportion: 86.8 , Maximum class 0 proportion: 94.8


Hence, we can see that the proportion coming from each class differs between different runs.

We could try stratified sampling instead:

In [6]:
# 导入Scikit-learn中的训练集与测试集划分函数
from sklearn.model_selection import train_test_split

# 创建空列表，用于保存50次分层抽样中类别0的百分比
water_df_stratified_sampling = []

# 使用1到50作为随机种子，重复分层抽样50次
for seed in range(1,51):

    # 从water_df中抽取250条记录
    # stratify按照potability的原始类别比例进行分层抽样
    # random_state固定本次抽样的随机种子
    # x1保存抽取的250条记录，_接收剩余记录
    x1, _ = train_test_split(water_df, train_size=250, test_size=None, stratify=water_df['potability'], random_state=seed)

    # 计算抽样后各类别的百分比，并提取类别0的百分比
    # 将本次抽样结果添加到列表中
    water_df_stratified_sampling.append((x1["potability"].value_counts(normalize=True).sort_index()*100).iloc[0])

# 输出50次分层抽样的类别0百分比
print(water_df_stratified_sampling)

# 输出50次抽样中类别0百分比的最小值与最大值
# round(..., 2)表示保留两位小数
print("Minimum class 0 proportion:", round(min(water_df_stratified_sampling),2), ", Maximum class 0 proportion:", round(max(water_df_stratified_sampling),2))

[np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4), np.float64(92.4)]
Minimum class 0 proportion: 92.4 , Maximum class 0 proportion: 92.4


So, what is happening above?:
+ We are iterating through 50 different seed values.
+ For each seed value, we are creating a training set of size 250 from the Water Potability dataset.
+ The "stratify" parameter ensures that in the produced split, the proportion of values will be equivalent to the proportions in water_df['potability'] (for example, if 25% of the values in water_df['potability'] had a value of 0, so will x1['potability']).
+ We append to water_df_stratified_sampling the proportion of each x1 having a potability value of 0.
+ We print the range of proportions having a potability value of 0. You will see that this is consistent across all sets (92.4% potability value of 0 in all sets).

Q1: Why is the potability value of 0 equal across all 50 sets - why is there no variability at all?

## Post-stratification

Let's extract a highly imbalanced sample from the water potability dataset.

In [7]:
# 计算原始数据集中potability各类别的比例
# normalize=True返回比例，而不是样本数量
population_proportions = water_df["potability"].value_counts(normalize=True)

# 输出原始总体的类别比例
# \n表示换行
print("True population target:\n", population_proportions)

# 从类别0中随机抽取2500条，从类别1中随机抽取7500条
# pd.concat()将两部分数据按行合并
# sample(frac=1)将合并后的全部记录随机打乱
# random_state=1保证抽样和打乱过程可复现
biased_water_df = pd.concat([water_df[water_df['potability']==0].sample(n=2500, random_state=1), water_df[water_df['potability']==1].sample(n=7500, random_state=1)]).sample(frac=1, random_state=1)

# 计算人为构造的数据集中各类别的百分比
print(biased_water_df["potability"].value_counts(normalize=True)*100)

# 保存人为构造的数据集中的类别比例
# 这里保存的是0到1之间的比例，没有乘以100
biased_sample_proportions = biased_water_df["potability"].value_counts(normalize=True)

True population target:
 potability
0    0.92384
1    0.07616
Name: proportion, dtype: float64
potability
1    75.0
0    25.0
Name: proportion, dtype: float64


In biased_water_df, we are intentionally choosing a sample with 2500 objects with a potability value of 0, and 7500 objects with a potability value of 1. Recall that this is very different to our original datasets class balance, where 92% of its objects had a class value of 0! We combine our two samples (2500 and 7500) using pd.concat(), and then shuffle using sample() (to avoid all of the objects where potability=0 being listed first, followed by all objects where potability=1, as this can cause problems in some data analytics approaches).

As discussed in lectures, we can calculate our sample weights as follows:

In [8]:
# 计算事后分层权重
# 权重 = 原始总体中该类别的比例 / 偏差样本中该类别的比例
# 类别在偏差样本中被低估时，权重大于1
# 类别在偏差样本中被高估时，权重小于1
post_stratification_weights = {0: population_proportions[0] / biased_sample_proportions[0], 1: population_proportions[1] / biased_sample_proportions[1]}

# 输出类别0和类别1对应的权重
print(post_stratification_weights)

# map()根据potability的类别标签查找对应权重
# 将每条记录的权重保存到新列sample_weight
biased_water_df['sample_weight'] = biased_water_df["potability"].map(post_stratification_weights)

# 输出添加了sample_weight列的数据集
print(biased_water_df)

{0: np.float64(3.69536), 1: np.float64(0.10154666666666667)}
         ph  hardness     tds  chlorine  sulfate  conductivity  \
57315  6.70    398.75  298.03      1.91    74.95       1747.92   
9018   6.28     93.69  233.43      0.60   219.11       1143.94   
20804  8.45    300.95  204.58      2.15   204.58       1042.32   
77432  8.12    156.70  294.41      0.88   192.48       1455.64   
43230  6.18     54.54   80.93      3.77   107.41       1294.21   
...     ...       ...     ...       ...      ...           ...   
82749  6.87    138.59  140.85      2.68   285.78        684.11   
1823   6.61    302.96  133.92      3.66   156.49        660.73   
12380  8.03    373.45  270.87      3.33   248.67       1985.07   
6976   8.33    327.32  268.00      2.36    57.38        490.30   
79881  7.68    235.29  301.42      7.43   322.92       2399.20   

       organic_carbon  trihalomethanes  turbidity  potability  sample_weight  
57315            3.93            23.57       2.53           1      

So, we should have a weight of around 0.1 for those with a potability value of 1, and 3.7 for those with a potability value of 0. We use .map() to create a column named sample_weight in our original biased_water_df data frame. This vector of sample weights can be passed to a machine learning model during training, allowing the machine learning model to recognise that the sample is not representative of the population, allowing it to weight each object accordingly.

## Part 2: Data Partitioning

This part will focus on data partitioning into training, testing and validation sets. There are some very useful existing libraries that we can use here.

One of the most common and useful functions here is one we've already used earlier, scikit-learn's train_test_split. This function takes in our dataset, and divides it into training and testing sets for us. It provides us with lots of useful opportunities for setting up our sets with the exact arrangement that we'd like (size of each set, the random state, and more).

Let's use the Bank Marketing dataset for this exercise, which can also be found on the Moodle.

In [9]:
bank_df = pd.read_csv('bank.csv')
print(bank_df.shape)

(11162, 17)


We can see that the Bank Marketing dataset contains over 11,000 objects. Let's use 80% of our data for training, and the other 20% for testing.

The first thing that we should do is divide our data into input features ($X$) and output targets ($y$). In the Bank dataset, we typically predict the "deposit" attribute, so we will consider that to be our target. Note that $X$ is usually capitalised as it is a matrix of features, whereas $y$ is usually not capitalised as it is a vector of values.

In [ ]:
# 提取deposit列作为目标变量y
# y通常是一维Series，用于表示模型需要预测的结果
y = bank_df["deposit"]

# 删除deposit列，保留其余所有列作为输入特征X
# X通常是二维DataFrame，包含模型用于预测的特征
X = bank_df.drop(columns=["deposit"])

Now that we have our data divided into input features ($X$) and output targets ($y$), we can form our training and testing sets:

In [11]:
# 将特征X和目标y划分为训练集和测试集
# test_size=0.2：20%的样本用于测试，80%用于训练
# random_state=1：固定随机种子，使每次划分结果一致
# X_train、X_test分别保存训练和测试特征
# y_train、y_test分别保存对应的真实目标值
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

Please note that there is a degree of randomness involved when using train_test_split to divide your data into training and testing sets. For repeatability and reproducability, it's important that you set a random_state value here. This will allow you to achieve the same training and testing sets if you were to carry out your experiment again with the same random state (sometimes called a random seed).

Let's have a look at our data:

In [12]:
print("======X_train======")
print(X_train.head())
print("Number of objects:", len(X_train))

print("======y_train======")
print(y_train.head())
print("Number of objects:", len(y_train))

======X_train======
       age          job  marital  education default  balance housing loan  \
8169    41  blue-collar   single  secondary      no     3851      no   no   
4779    38  blue-collar   single   tertiary      no     2885     yes   no   
5476    34  blue-collar  married   tertiary      no      748     yes  yes   
4766    34       admin.  married   tertiary      no       69      no  yes   
11044   32     services  married  secondary      no        0      no   no   

        contact  day month  duration  campaign  pdays  previous poutcome  
8169   cellular   19   nov       130         1     -1         0  unknown  
4779    unknown   20   jun      1422        10     -1         0  unknown  
5476   cellular    6   may       161         1     -1         0  unknown  
4766   cellular   25   nov       296         1    205         2  failure  
11044  cellular   31   jul       105         6     -1         0  unknown  
Number of objects: 8929
======y_train======
8169      no
4779     y

You should see that $X\_train$ and $y\_train$ both have a length of 8,929, whereas $X\_test$ and $y\_test$ should have a length of 2,233 (see below).

In [13]:
print("======X_test======")
print(X_test.head())
print(len(X_test))

print("======y_test======")
print(y_test.head())
print(len(y_test))

======X_test======
      age          job  marital  education default  balance housing loan  \
8422   31       admin.   single  secondary      no      554     yes   no   
9226   43  blue-collar  married  secondary      no      514     yes  yes   
8162   26  blue-collar   single  secondary      no      277      no   no   
4337   39     services   single  secondary      no       91      no   no   
2438   40   management  married   tertiary      no        0      no   no   

       contact  day month  duration  campaign  pdays  previous poutcome  
8422  cellular    5   feb        94         1     -1         0  unknown  
9226  cellular   20   apr        34         2     -1         0  unknown  
8162   unknown   14   may       514         2     -1         0  unknown  
4337  cellular    3   sep       213         1    185         1  success  
2438  cellular    6   aug       131         2     98         6  success  
2233
======y_test======
8422     no
9226     no
8162     no
4337    yes
2438    

We could go ahead and train and test our model right now. However, we should also prepare a validation set. Unfortunately, scikit-learn does not have a function which can split a dataset into training, testing, and validation sets. We can instead choose to call train_test_split twice to achieve our desired outcome.

In [ ]:
# 定义训练集、验证集、测试集占原始数据集的比例
train_proportion = 0.75
validation_proportion = 0.15
test_proportion = 0.1

# 第一次划分：75%作为训练集，剩余25%暂存到X_test和y_test
# 注意：这里的X_test和y_test还不是最终测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_proportion+validation_proportion), random_state=1)

# 第二次划分：将剩余25%的数据进一步拆分
# 其中60%作为验证集，40%作为最终测试集
# 相对于原始数据，分别对应15%和10%
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=test_proportion/(test_proportion + validation_proportion), random_state=1)

# 查看原始数据集的维度
print("======Overall Dataset Shape======")
print(X.shape)
print(y.shape)

# 查看训练集的维度
print("======Training Set Shape======")
print(X_train.shape)
print(y_train.shape)

# 查看测试集的维度
print("======Testing Set Shape======")
print(X_test.shape)
print(y_test.shape)

# 查看验证集的维度
print("======Validation Set Shape======")
print(X_val.shape)
print(y_val.shape)

======Overall Dataset Shape======
(11162, 16)
(11162,)
======Training Set Shape======
(8371, 16)
(8371,)
======Testing Set Shape======
(1117, 16)
(1117,)
======Validation Set Shape======
(1674, 16)
(1674,)


You should be able to see from above that our dataset, which has 11,162 objects, has been split into a training set of 8,371 objects, a testing set of 1,117 objects, and a validation set of 1,674 objects.

Now, let's look at a full example of where validation data can be used to support model training. For this example, we are going to use a decision tree classifier, so the first thing that we need to do is import it. Also, let's choose a random seed and create a random seed variable ($seed$), rather than hard-coding our random seed every time we wish to use it:

In [16]:
# 从Scikit-learn的tree模块中导入决策树分类器
from sklearn.tree import DecisionTreeClassifier

# 定义随机种子为3，使后续随机操作可复现
seed = 3

In [17]:
bank_df = pd.read_csv("bank.csv")
y = bank_df['deposit'].map({"yes": 1, 'no': 0}) #Labels are currently in the format "yes" and "no". Map them onto 0 and 1.
X = bank_df.drop(columns=['deposit']) #All of the dataset excluding the labels.
X = pd.get_dummies(X, drop_first=True)

train_proportion = 0.75
validation_proportion = 0.15
test_proportion = 0.1
#The proportions of data we're using for training, validation and testing respectively.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_proportion+validation_proportion), random_state=seed)
#Divide into 1) training and 2) testing (where the testing includes validation TEMPORARILY - it will be divided in the next step).

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=test_proportion/(test_proportion + validation_proportion), random_state=seed)
#Split the testing into testing and validation.

clf_baseline = DecisionTreeClassifier(random_state=seed).fit(X_train, y_train)
#Build a "baseline" decision tree using the training data.


The above code shows a simple pipeline for training an ML model - 1) import data, 2) preprocess data, 3) split into training/testing/validation sets, and 4) train the model. What's missing, is 1) utilising the validation set and 2) evaluating the model. When training some ML models, the validation set will be used during the training stage. Note, this does not mean it is used *for* training, but is instead used for tasks such as early stopping (knowing when to stop training). In the case of a decision tree, the validation set can be used to determine how much pruning should be applied, which prevents overfitting on the training set.

In [18]:
# 读取银行客户数据集
bank_df = pd.read_csv("bank.csv")

# 提取deposit作为目标变量y
# map()将yes转换为1，将no转换为0
y = bank_df['deposit'].map({"yes": 1, 'no': 0})

# 删除目标变量deposit，保留其他列作为输入特征X
X = bank_df.drop(columns=['deposit'])

# pd.get_dummies()：
# 对类别型特征进行独热编码（One-Hot Encoding）
#
# 例如：
# job有student、admin、technician三个类别
#
# 原本：
# job = student
#
# 编码后可能得到：
# job_student = True
# job_technician = False
#
# drop_first=True：
# 对每个类别型变量省略一个基准类别
# 如果原来有k个类别，通常生成k-1个虚拟变量
#
# 作用：
# 避免为每个类别都保留完整的冗余指示列
# 在线性模型中有助于避免完全共线性
#
# 注意：
# 对决策树来说，drop_first=True并不是必要的
#
# 返回值：
# 新的DataFrame，类别型列被展开为数值/布尔特征列
X = pd.get_dummies(X, drop_first=True)

# 指定训练集、验证集和测试集的比例
train_proportion = 0.75
validation_proportion = 0.15
test_proportion = 0.1

# 第一次划分：75%作为训练集，25%作为临时数据集
# random_state=seed：使用之前定义的随机种子
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_proportion+validation_proportion), random_state=seed)

# 第二次划分：将剩余25%的数据分成15%验证集和10%测试集
# test_size=0.1/0.25=0.4，表示剩余数据的40%作为测试集
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=test_proportion/(test_proportion + validation_proportion), random_state=seed)

# 创建决策树分类器，并使用训练集拟合模型
# clf_baseline保存训练完成的基线模型
clf_baseline = DecisionTreeClassifier(random_state=seed).fit(X_train, y_train)

The ccp_alphas here can be considered thresholds that act like trimming settings for the tree. For example, a alpha value of 0 will lead to no change to the tree, a value of 0.05 will lead to some weak values being trimmed, a value of 0.1 will lead to slightly less weak values being trimmed, etc. Once we have obtained the alphas for our baseline tree, we then build smart trees which utilise these alphas to prune branches with little predictive power. We do this for all potential alphas, and at the end, keep the best tree.

We then use the test data to create a final evaluation of our model on unseen data. In many runs, you should see that the smart tree achieves a higher test score than the baseline tree, as pruning can prevent overfitting.

Q2: We have used one-hot encoding above (pd.get_dummies). What is this, and why is it necessary? What is the drop_first parameter doing?  
Q3: Download the Learning from Imbalanced Insurance dataset (insurance.csv) from Moodle. It is highly imbalanced, with 84% of objects in class 0, and 16% in class 1. Let's assume the population proportions are 92% and 8% respectively. Split the data into training and testing sets, calculate sample weights, and train a RandomForestClassifier. Evaluate two models, one which incorporates sample weights, and one which does not, using scikit-learn's classification_report method and sample_weights parameter.  
Q4: If you finish early, you may want to spend the remaining lab time working on your coursework.

## Bibliography

1. Scikit-learn, Undated. Post pruning decision trees with cost complexity pruning. Available at: https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html (Accessed 09 November 2026).
2. S. Moro, P. Cortez and P. Rita. A Data-Driven Approach to Predict the Success of Bank Telemarketing. Decision Support Systems, Elsevier, 62:22-31, June 2014.
3. DeveloperGhost, Undated. Water Potability. Available at: https://www.kaggle.com/datasets/developerghost/water-potability (Accessed 13 November 2026). Data licence is CC BY-NC-SA 4.0 (https://creativecommons.org/licenses/by-nc-sa/4.0/).
4. Dataset downloaded from: Kaggle, Undated. Water Potability. Available at: https://www.kaggle.com/datasets/developerghost/water-potability (Accessed 13 November 2026).
5. Dataset downloaded from: Kaggle, Undated. Bank Marketing Dataset. Available at: https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset (Accessed 13 November 2026).
6. Dataset downloaded from: Kaggle, Undated. Learning from Imbalanced Insurance Data. Available at: https://www.kaggle.com/datasets/arashnic/imbalanced-data-practice (Accessed 13 November 2026).